# Лабораторная работа № 3

Выполнил: студент гр. 6403-010302D Андрей Баландин

In [2]:
import sys
from pyspark.sql import SparkSession, functions as sf
from pyspark.sql.window import Window

Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

In [6]:
# Инициализация Spark сессии
spark = SparkSession.builder.appName("StackOverflow_Analysis").getOrCreate()

# 1. Загрузка языков программирования из CSV
languages_df = spark.read.csv("/content/programming-languages.csv", header=True, inferSchema=True)
# Приведение названий языков к нижнему регистру для удобства сопоставления
languages_list = [row['name'].lower() for row in languages_df.select("name").collect()]

print(f"Загружено {len(languages_list)} языков программирования.")

Загружено 700 языков программирования.


In [7]:
import xml.etree.ElementTree as ET

def parse_xml_row(xml_string):
    try:
        # Простая логика парсинга для Python с использованием xml.etree
        root = ET.fromstring(xml_string)
        return (
            root.attrib.get('CreationDate'),
            root.attrib.get('Tags')
        )
    except:
        return None

# 3. Обработка posts_sample.xml
# Загрузка как RDD
posts_raw_rdd = spark.sparkContext.textFile("/content/posts_sample.xml")

# Добавление индекса и фильтрация первых двух и последней строк
indexed_rdd = posts_raw_rdd.zipWithIndex()
count = indexed_rdd.count()
filtered_rdd = indexed_rdd.filter(lambda x: x[1] > 1 and x[1] < count - 1).map(lambda x: x[0])

# Парсинг XML и извлечение CreationDate и Tags
parsed_data_rdd = filtered_rdd.map(parse_xml_row).filter(lambda x: x is not None and x[1] is not None)

# Преобразование в DataFrame
posts_df = parsed_data_rdd.toDF(["CreationDate", "Tags"])

# Обработка данных: извлечение года и поиск языков в тегах
posts_df = posts_df.withColumn("Year", sf.year(sf.col("CreationDate")))
posts_df = posts_df.filter((sf.col("Year") >= 2010) & (sf.col("Year") <= 2020))

# Функция для определения того, какие языки из нашего списка присутствуют в строке тегов
def extract_langs(tags_str):
    if not tags_str:
        return []
    # Теги отформатированы как <java><spring>
    tags = tags_str.strip('<>').split('><')
    return [t for t in tags if t.lower() in languages_list]

extract_langs_udf = sf.udf(extract_langs, sf.ArrayType(sf.StringType()))

# Развертывание тегов, чтобы каждый язык в посте получил свою строку
final_df = posts_df.withColumn("Language", sf.explode(extract_langs_udf(sf.col("Tags"))))

# Расчет Топ-10 для каждого года
window_spec = Window.partitionBy("Year").orderBy(sf.col("count").desc())

report_df = final_df.groupBy("Year", "Language").count() \
    .withColumn("rank", sf.row_number().over(window_spec)) \
    .filter(sf.col("rank") <= 10) \
    .orderBy("Year", "rank")

# Отображение результатов
display(report_df.toPandas())

# Сохранение в Parquet
report_df.write.mode("overwrite").parquet("top_languages_2010_2020.parquet")
print("Отчет сохранен в top_languages_2010_2020.parquet")

,Year,Language,count,rank
0,2010,php,6,1
1,2010,javascript,4,2
2,2010,c,3,3
3,2010,python,3,4
4,2010,java,2,5
...,...,...,...,...
95,2019,matlab,2,6
96,2019,go,1,7
97,2019,r,1,8
98,2019,curl,1,9


Отчет сохранен в top_languages_2010_2020.parquet
